In [48]:
# Imports and Setup
import psycopg2
import psycopg2.extensions
import select
import json
import logging
import os
from datetime import datetime
from typing import Dict, Any
from pathlib import Path
import yaml
import pandas as pd
import time
import sys

# Add the src directory to Python path
notebook_path = Path().absolute()  # Get current notebook directory
src_path = notebook_path.parent    # Go up one level to src
sys.path.append(str(src_path))

print(f"Current path: {notebook_path}")
print(f"Source path: {src_path}")

# Now import the Indicator class
from indicator.indicator import Indicator

# Create logs directory if it doesn't exist
log_dir = notebook_path.parent.parent / 'logs'
log_dir.mkdir(exist_ok=True)
log_file = log_dir / 'kline_listener.log'

# Rest of your code remains the same...

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class KlineListener:
    def __init__(self, source_db_params: Dict[str, str], target_db_params: Dict[str, str], 
                 source_table: str = None, target_table: str = None):
        """
        Initialize connections to both source and target databases.
        
        Args:
            source_db_params: Connection parameters for BitunixData database
            target_db_params: Connection parameters for Indicator database
            source_table: Name of the source table to listen to (e.g., 'btcusdt_1')
            target_table: Name of the target table for indicators (e.g., 'btcusdt_1_indicators')
        """
        self.source_db_params = source_db_params
        self.target_db_params = target_db_params
        self.source_conn = None
        self.target_conn = None
        
        # Store table names
        self.source_table = source_table
        self.target_table = target_table or (f"{source_table}_indicators" if source_table else None)
        
        # Store table metadata
        self.symbol = None
        self.timeframe = None
        if source_table:
            # Parse symbol and timeframe from table name (e.g., 'btcusdt_1' -> 'btc', '1')
            parts = source_table.split('_')
            if len(parts) == 2:
                self.symbol = parts[0].replace('usdt', '')  # btcusdt -> btc
                self.timeframe = parts[1]
    
    def set_tables(self, source_table: str, target_table: str = None) -> None:
        """
        Set or update the source and target tables.
        
        Args:
            source_table: Name of source table
            target_table: Optional name of target table (will use source_table_indicators if not provided)
        """
        self.source_table = source_table
        self.target_table = target_table or f"{source_table}_indicators"
        
        # Update symbol and timeframe
        parts = source_table.split('_')
        if len(parts) == 2:
            self.symbol = parts[0].replace('usdt', '')
            self.timeframe = parts[1]
    
    def get_historical_data(self, end_ts: datetime, limit: int = 50) -> pd.DataFrame:
        """
        Get historical data from source table.
        
        Args:
            end_ts: End timestamp for the data
            limit: Number of rows to fetch
        
        Returns:
            DataFrame with historical data
        """
        if not self.source_table:
            raise ValueError("Source table not set")
            
        with self.source_conn.cursor() as cur:
            cur.execute(f"""
                SELECT id, ts, CAST(open AS FLOAT), CAST(high AS FLOAT), 
                       CAST(low AS FLOAT), CAST(close AS FLOAT)
                FROM {self.source_table}
                WHERE ts <= %s
                ORDER BY ts DESC
                LIMIT %s
            """, (end_ts, limit))
            
            data = cur.fetchall()
            df = pd.DataFrame(data[::-1], 
                            columns=['id', 'ts', 'open', 'high', 'low', 'close'])
            df['ts'] = pd.to_datetime(df['ts'])
            return df
        
    def connect(self) -> None:
        """Establish database connections"""
        try:
            # Connect to source database (BitunixData)
            self.source_conn = psycopg2.connect(**self.source_db_params)
            self.source_conn.set_isolation_level(psycopg2.extensions.ISOLATION_LEVEL_AUTOCOMMIT)
            
            # Connect to target database (Indicator)
            self.target_conn = psycopg2.connect(**self.target_db_params)
            self.target_conn.autocommit = True
            
            logger.info("Successfully connected to both databases")
        except Exception as e:
            logger.error(f"Error connecting to databases: {str(e)}")
            raise

    def process_notification(self, payload: Dict[str, Any]) -> None:
        """Process notification and calculate indicators"""
        try:
            # Extract data from payload
            source_table = payload['table']
            source_table = source_table.replace('test_', '')
            print(f"(Process_Notification) Source table: {source_table}")
            symbol = payload['symbol']
            timeframe = self._normalize_timeframe(payload['timeframe'])
            
            # Construct indicator table name
            indicator_table = f"{source_table}_indicators"
            
            logger.info(f"(Process_Notification) Processing {symbol} {timeframe} data update")
            
            # Ensure table exists
            self._ensure_indicator_table(indicator_table)
            
            print(f"(Process_Notification) Indicator table: {indicator_table}")
            # Get historical data for calculations
            with self.source_conn.cursor() as source_cur:
                source_cur.execute(f"""
                    SELECT id, ts, 
                        CAST(open AS FLOAT), CAST(high AS FLOAT), 
                        CAST(low AS FLOAT), CAST(close AS FLOAT)
                    FROM {source_table}
                    WHERE ts <= %s
                    ORDER BY ts DESC
                    LIMIT 50
                """, (payload['data']['ts'],))
                
                historical_data = source_cur.fetchall()
                print(f"(Process_Notification) Historical data: {historical_data}")
                
                if len(historical_data) < 20:
                    logger.info(f"Insufficient historical data ({len(historical_data)} rows). Skipping.")
                    return
                
                # Create DataFrame with historical data (reverse to get ascending order)
                df = pd.DataFrame(historical_data[::-1], columns=['id', 'ts', 'open', 'high', 'low', 'close'])
                df['ts'] = pd.to_datetime(df['ts'])
                
                # Ensure numeric columns are float
                numeric_columns = ['open', 'high', 'low', 'close']
                df[numeric_columns] = df[numeric_columns].astype(float)
                
                # Calculate indicators
                indicator = Indicator(df, symbol=symbol, timeframe=timeframe)
                
                try:
                    indicator.ema(period=14)
                    indicator.rsi(period=14)
                    indicator.bollinger_bands()
                    indicator.stoch_rsi()
                    indicator.atr()
                    indicator.cci()
                    indicator.roc(period=12)
                    indicator.momentum(period=14)
                    indicator.parabolic_sar()
                    indicator.williams_r(period=14)
                except Exception as e:
                    logger.error(f"Error calculating indicators: {str(e)}")
                    return
                
                # Get the last row for insertion (most recent)
                results_df = indicator.df.iloc[-1:].copy()
                
                # Insert the calculated row
                with self.target_conn.cursor() as target_cur:
                    row = results_df.iloc[0]
                    target_cur.execute(f"""
                        INSERT INTO {indicator_table} (
                            source_id, ts, open, high, low, close,
                            ema_14, rsi_14, bb_upper, bb_middle, bb_lower,
                            stoch_rsi, stoch_k, stoch_d, atr, cci,
                            roc_12, momentum_14, psar, williams_r_14
                        ) VALUES (
                            %s, %s, %s, %s, %s, %s,
                            %s, %s, %s, %s, %s,
                            %s, %s, %s, %s, %s,
                            %s, %s, %s, %s
                        )
                        ON CONFLICT (source_id) DO UPDATE SET
                            ts = EXCLUDED.ts,
                            open = EXCLUDED.open,
                            high = EXCLUDED.high,
                            low = EXCLUDED.low,
                            close = EXCLUDED.close,
                            ema_14 = EXCLUDED.ema_14,
                            rsi_14 = EXCLUDED.rsi_14,
                            bb_upper = EXCLUDED.bb_upper,
                            bb_middle = EXCLUDED.bb_middle,
                            bb_lower = EXCLUDED.bb_lower,
                            stoch_rsi = EXCLUDED.stoch_rsi,
                            stoch_k = EXCLUDED.stoch_k,
                            stoch_d = EXCLUDED.stoch_d,
                            atr = EXCLUDED.atr,
                            cci = EXCLUDED.cci,
                            roc_12 = EXCLUDED.roc_12,
                            momentum_14 = EXCLUDED.momentum_14,
                            psar = EXCLUDED.psar,
                            williams_r_14 = EXCLUDED.williams_r_14
                    """, (
                        int(row['id']), row['ts'], 
                        float(row['open']), float(row['high']), 
                        float(row['low']), float(row['close']),
                        float(row.get('ema_14', 0) or 0), 
                        float(row.get('rsi_14', 0) or 0),
                        float(row.get('bb_upper', 0) or 0), 
                        float(row.get('bb_middle', 0) or 0), 
                        float(row.get('bb_lower', 0) or 0),
                        float(row.get('stoch_rsi', 0) or 0), 
                        float(row.get('stoch_k', 0) or 0), 
                        float(row.get('stoch_d', 0) or 0),
                        float(row.get('atr', 0) or 0), 
                        float(row.get('cci', 0) or 0),
                        float(row.get('roc_12', 0) or 0), 
                        float(row.get('momentum_14', 0) or 0),
                        float(row.get('psar', 0) or 0), 
                        float(row.get('williams_r_14', 0) or 0)
                    ))
                    
                self.target_conn.commit()
                logger.info(f"Successfully processed new row for {symbol} {timeframe}")
                
        except Exception as e:
            logger.error(f"Error processing notification: {str(e)}")
            logger.error(f"Payload: {payload}")
            self.target_conn.rollback()


    def create_indicator_table(self, table_name: str) -> None:
        """Create indicator table if it doesn't exist"""
        try:
            with self.target_conn.cursor() as cur:
                # First check if table exists
                cur.execute("""
                    SELECT EXISTS (
                        SELECT FROM information_schema.tables 
                        WHERE table_schema = 'public' 
                        AND table_name = %s
                    )
                """, (table_name,))
                
                exists = cur.fetchone()[0]
                
                if not exists:
                    cur.execute(f"""
                        CREATE TABLE {table_name} (
                            id SERIAL PRIMARY KEY,
                            source_id INTEGER NOT NULL,
                            ts TIMESTAMP NOT NULL,
                            open NUMERIC(20, 8),
                            high NUMERIC(20, 8),
                            low NUMERIC(20, 8),
                            close NUMERIC(20, 8),
                            ema_14 NUMERIC(20, 8),
                            rsi_14 NUMERIC(20, 8),
                            bb_upper NUMERIC(20, 8),
                            bb_middle NUMERIC(20, 8),
                            bb_lower NUMERIC(20, 8),
                            stoch_rsi NUMERIC(20, 8),
                            stoch_k NUMERIC(20, 8),
                            stoch_d NUMERIC(20, 8),
                            atr NUMERIC(20, 8),
                            cci NUMERIC(20, 8),
                            roc_12 NUMERIC(20, 8),
                            momentum_14 NUMERIC(20, 8),
                            psar NUMERIC(20, 8),
                            williams_r_14 NUMERIC(20, 8),
                            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                            UNIQUE(source_id)
                        )
                    """)
                    self.target_conn.commit()
                    
                    # Verify table was created
                    cur.execute("""
                        SELECT EXISTS (
                            SELECT FROM information_schema.tables 
                            WHERE table_schema = 'public' 
                            AND table_name = %s
                        )
                    """, (table_name,))
                    
                    if not cur.fetchone()[0]:
                        raise Exception(f"Failed to create table {table_name}")
                    
                    logger.info(f"Created indicator table: {table_name}")
                    
        except Exception as e:
            self.target_conn.rollback()
            logger.error(f"Error creating indicator table {table_name}: {str(e)}")
            raise

    def _normalize_timeframe(self, timeframe: str) -> str:
        """Convert timeframe to standard format"""
        # Map common variations to standard format
        timeframe_map = {
            '1': '1m',
            '3': '3m',
            '5': '5m',
            '15': '15m',
            '30': '30m',
            '60': '1h',
            '60m': '1h',
            '120': '2h',
            '240': '4h',
            '360': '6h',
            '720': '12h',
            '1440': '1d'
        }
        
        # If timeframe is already in correct format, return it
        if timeframe in ['1m', '3m', '5m', '15m', '30m', '1h', '2h', '4h', '6h', '12h', '1d']:
            return timeframe
        
        # Try to get from map, otherwise append 'm'
        return timeframe_map.get(timeframe, f"{timeframe}m")


    def _ensure_indicator_table(self, table_name: str) -> None:
        """Ensure indicator table exists"""
        with self.target_conn.cursor() as cur:
            cur.execute(f"""
                CREATE TABLE IF NOT EXISTS {table_name} (
                    id SERIAL PRIMARY KEY,
                    source_id INTEGER UNIQUE NOT NULL,
                    ts TIMESTAMP NOT NULL,
                    open NUMERIC(20, 8),
                    high NUMERIC(20, 8),
                    low NUMERIC(20, 8),
                    close NUMERIC(20, 8),
                    ema_14 NUMERIC(20, 8),
                    rsi_14 NUMERIC(20, 8),
                    bb_upper NUMERIC(20, 8),
                    bb_middle NUMERIC(20, 8),
                    bb_lower NUMERIC(20, 8),
                    stoch_rsi NUMERIC(20, 8),
                    stoch_k NUMERIC(20, 8),
                    stoch_d NUMERIC(20, 8),
                    atr NUMERIC(20, 8),
                    cci NUMERIC(20, 8),
                    roc_12 NUMERIC(20, 8),
                    momentum_14 NUMERIC(20, 8),
                    psar NUMERIC(20, 8),
                    williams_r_14 NUMERIC(20, 8),
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """)
            self.target_conn.commit()

    def listen(self) -> None:
        """Start listening for notifications"""
        try:
            with self.source_conn.cursor() as cur:
                cur.execute("LISTEN kline_updates;")
                logger.info("Started listening for kline updates...")
                
                while True:
                    if select.select([self.source_conn], [], [], 5) != ([], [], []):
                        self.source_conn.poll()
                        while self.source_conn.notifies:
                            notify = self.source_conn.notifies.pop(0)
                            payload = json.loads(notify.payload)
                            self.process_notification(payload)
                            
        except Exception as e:
            logger.error(f"Error in listener: {str(e)}")
            raise
        finally:
            self.cleanup()

    def cleanup(self) -> None:
        """Clean up database connections"""
        if self.source_conn:
            self.source_conn.close()
        if self.target_conn:
            self.target_conn.close()
        logger.info("Cleaned up database connections")


Current path: c:\My Folder\CryptoAPI\src\listener
Source path: c:\My Folder\CryptoAPI\src


In [40]:
# Test Cell: Create Test Tables
def create_test_tables(conn):
    """Create test tables for 1m and 240m timeframes with the same structure as production tables."""
    try:
        with conn.cursor() as cur:
            # Create test_btcusdt_1 table
            cur.execute("""
                CREATE TABLE IF NOT EXISTS test_btcusdt_1 (
                    id SERIAL PRIMARY KEY,
                    source_id INTEGER NOT NULL,
                    ts TIMESTAMP NOT NULL,
                    open NUMERIC(20, 8),
                    high NUMERIC(20, 8),
                    low NUMERIC(20, 8),
                    close NUMERIC(20, 8),
                    ema_14 NUMERIC(20, 8),
                    rsi_14 NUMERIC(20, 8),
                    bb_upper NUMERIC(20, 8),
                    bb_middle NUMERIC(20, 8),
                    bb_lower NUMERIC(20, 8),
                    stoch_rsi NUMERIC(20, 8),
                    stoch_k NUMERIC(20, 8),
                    stoch_d NUMERIC(20, 8),
                    atr NUMERIC(20, 8),
                    cci NUMERIC(20, 8),
                    roc_12 NUMERIC(20, 8),
                    momentum_14 NUMERIC(20, 8),
                    psar NUMERIC(20, 8),
                    williams_r_14 NUMERIC(20, 8),
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    UNIQUE(source_id)
                )
            """)

            # Create test_btcusdt_240 table
            cur.execute("""
                CREATE TABLE IF NOT EXISTS test_btcusdt_240 (
                    id SERIAL PRIMARY KEY,
                    source_id INTEGER NOT NULL,
                    ts TIMESTAMP NOT NULL,
                    open NUMERIC(20, 8),
                    high NUMERIC(20, 8),
                    low NUMERIC(20, 8),
                    close NUMERIC(20, 8),
                    ema_14 NUMERIC(20, 8),
                    rsi_14 NUMERIC(20, 8),
                    bb_upper NUMERIC(20, 8),
                    bb_middle NUMERIC(20, 8),
                    bb_lower NUMERIC(20, 8),
                    stoch_rsi NUMERIC(20, 8),
                    stoch_k NUMERIC(20, 8),
                    stoch_d NUMERIC(20, 8),
                    atr NUMERIC(20, 8),
                    cci NUMERIC(20, 8),
                    roc_12 NUMERIC(20, 8),
                    momentum_14 NUMERIC(20, 8),
                    psar NUMERIC(20, 8),
                    williams_r_14 NUMERIC(20, 8),
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    UNIQUE(source_id)
                )
            """)

            conn.commit()
            print("Successfully created test tables: test_btcusdt_1 and test_btcusdt_240")

    except Exception as e:
        conn.rollback()
        print(f"Error creating test tables: {str(e)}")
        raise

# Execute the function
# try:
#     # Use the source connection from your KlineListener
#     test_listener = KlineListener(source_params, target_params)
#     test_listener.connect()
    
#     create_test_tables(test_listener.source_conn)
    
#     # Optional: Verify the tables were created
#     with test_listener.source_conn.cursor() as cur:
#         cur.execute("""
#             SELECT table_name 
#             FROM information_schema.tables 
#             WHERE table_name LIKE 'test_btcusdt_%'
#         """)
#         tables = cur.fetchall()
#         print("\nCreated tables:")
#         for table in tables:
#             print(f"- {table[0]}")
            
# except Exception as e:
#     print(f"Setup error: {str(e)}")
# finally:
#     test_listener.cleanup()

In [41]:
# Test Cell: Process Sample Notification
# Setup paths and load configs


# Test different timeframe scenarios
test_payloads = [
    {
        'table': 'test_btcusdt_1',  # Table name in source database
        'symbol': 'btc',        # Cryptocurrency symbol
        'timeframe': '1',       # Original timeframe value
        'operation': 'INSERT',  # Database operation
        'timestamp': datetime.now().isoformat(),  # Current timestamp
        'data': {
            'id': 1001,        # Source row ID
            'ts': datetime.now().isoformat(),  # Kline timestamp
            'open': 50123.45,
            'high': 50234.56,
            'low': 50012.34,
            'close': 50123.45
        }
    },
    {
        'table': 'test_btcusdt_240',  # 4h table
        'symbol': 'btc',
        'timeframe': '240',     # Should be normalized to '4h'
        'operation': 'INSERT',
        'timestamp': datetime.now().isoformat(),
        'data': {
            'id': 1002,
            'ts': datetime.now().isoformat(),
            'open': 50123.45,
            'high': 50234.56,
            'low': 50012.34,
            'close': 50123.45
        }
    }
]


In [49]:
notebook_path = Path().absolute()
config_dir = notebook_path.parent / 'config'

with open(config_dir / 'database_config.yaml', 'r') as f:
    source_params = yaml.safe_load(f)
    
with open(config_dir / 'indicator_config.yaml', 'r') as f:
    target_params = yaml.safe_load(f)

# Initialize listener
test_listener = KlineListener(source_params, target_params)
test_listener.connect()

# Process each test payload
try:
    for payload in test_payloads:
        # Modify payload to read from production table but write to test table
        read_table = payload['table'].replace('test_', '')
        payload['source_table'] = read_table  # Table to read from
        payload['target_table'] = f"test_{read_table}"  # Table to write to
        print(f"Payload: {payload}")
        print(f"\nProcessing payload - Reading from: {read_table}, Writing to: {payload['target_table']}")
        print(f"Symbol: {payload['symbol']}, Timeframe: {payload['timeframe']}")
        test_listener.process_notification(payload)
except Exception as e:
    print(f"Error during test: {str(e)}")
finally:
    test_listener.cleanup()

2024-11-27 15:42:17,634 - INFO - Successfully connected to both databases
2024-11-27 15:42:17,634 - INFO - (Process_Notification) Processing btc 1m data update
2024-11-27 15:42:17,656 - INFO - Successfully processed new row for btc 1m
2024-11-27 15:42:17,656 - INFO - (Process_Notification) Processing btc 4h data update
2024-11-27 15:42:17,674 - INFO - Successfully processed new row for btc 4h
2024-11-27 15:42:17,674 - INFO - Cleaned up database connections


Payload: {'table': 'test_btcusdt_1', 'symbol': 'btc', 'timeframe': '1', 'operation': 'INSERT', 'timestamp': '2024-11-27T15:24:28.807269', 'data': {'id': 1001, 'ts': '2024-11-27T15:24:28.807269', 'open': 50123.45, 'high': 50234.56, 'low': 50012.34, 'close': 50123.45}, 'source_table': 'btcusdt_1', 'target_table': 'test_btcusdt_1'}

Processing payload - Reading from: btcusdt_1, Writing to: test_btcusdt_1
Symbol: btc, Timeframe: 1
(Process_Notification) Source table: btcusdt_1
(Process_Notification) Indicator table: btcusdt_1_indicators
(Process_Notification) Historical data: [(2989, datetime.datetime(2024, 11, 27, 15, 24), 94428.58, 94429.44, 94428.57, 94429.44), (2988, datetime.datetime(2024, 11, 27, 15, 23), 94405.7, 94467.71, 94392.95, 94428.58), (2987, datetime.datetime(2024, 11, 27, 15, 22), 94513.75, 94543.98, 94392.95, 94405.7), (2986, datetime.datetime(2024, 11, 27, 15, 21), 94619.04, 94619.05, 94488.0, 94520.92), (2985, datetime.datetime(2024, 11, 27, 15, 20), 94639.98, 94659.99,